# 🌍 Language Detection (17 Languages)


## 📌 Table of Contents
- [1. Business Overview & Goal](#1-business-overview--goal)
- [2. Imports & Setup](#2-imports--setup)
- [3. Load Data](#3-load-data)
- [4. EDA](#4-eda)
- [5. Feature Engineering](#5-feature-engineering)
- [6. Modeling & Evaluation](#6-modeling--evaluation)
- [7. Save Model](#7-save-model)

## 1. Business Overview & Goal

### Business Overview
We have a small text dataset containing **17 languages**. Each row includes a short text snippet and its language label.

### Goal
Build an NLP model that predicts the **language** of a given text among:
English, Malayalam, Hindi, Tamil, Kannada, French, Spanish, Portuguese, Italian, Russian, Sweedish, Dutch, Arabic, Turkish, German, Danish, Greek.

### Target
- **Target column:** `language` (label)
- **Input column:** `text` (raw string)
- **Metric:** Accuracy + per-class performance (classification report)

# 2. Import Libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import nltk 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

# 3. Reading Data

In [ ]:
df=pd.read_csv('Language Detection.csv')

In [ ]:
df.head()

# 4. EDA

In [ ]:
df.info()

In [ ]:
df['Language'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(12,6))
sns.countplot(data=df, y='Language', order=df['Language'].value_counts().index)
plt.title("Language Distribution")
plt.show()

In [ ]:
df['Text'][0]

In [ ]:
df.shape

# 5. Feature Engineering

In [ ]:
# Convert all text to lowercase
df['Text'] = df['Text'].str.lower()

# Remove punctuation characters
df['Text'] = df['Text'].str.replace(r'[^\w\s]', '', regex=True)

# Remove numbers
df['Text'] = df['Text'].str.replace(r'\d+', '', regex=True)

# Remove newline characters
df['Text'] = df['Text'].str.replace(r'\n', ' ', regex=True)

# Remove extra spaces
df['Text'] = df['Text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
df['Text'][0]

In [ ]:
def simple_tokenize(s: str):
    return s.split()

df["tokens"] = df["Text"].apply(simple_tokenize)
df[["Text", "tokens"]].head()

In [ ]:
df['Text'] = df['Text'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
x = df['Text']
y = df['Language']

In [ ]:
vect = CountVectorizer(ngram_range=(1,2))
x_vec = vect.fit_transform(x)

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x_vec,y,test_size=0.2,random_state=42)

# 6. Modelling& Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000)
model.fit(x_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score,classification_report

pred = model.predict(x_test)

print("Accuracy:",accuracy_score(y_test,pred))
print(classification_report(y_test,pred))

### 6.1 Confusion Matrix 

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(10,8))

sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

### Confusion Matrix

The confusion matrix visualizes the model's performance across all language classes.

Most predictions lie on the diagonal, indicating correct classifications.  
This shows that the model is able to successfully distinguish between the 17 different languages.

### 6.2 Demo

In [ ]:
def predict_language(text):
    
    text = text.lower()
    
    text_vec = vect.transform([text])   # vectorize
    
    prediction = model.predict(text_vec)
    
    return prediction[0]

In [ ]:
predict_language("Merhaba nasılsın bugün hava çok güzel")

In [ ]:
predict_language("Bonjour comment allez vous")

In [ ]:
predict_language("Привет как дела")

In [ ]:
examples = [
    "Merhaba nasılsın bugün hava çok güzel",
    "Hola amigo como estas",
    "Bonjour je suis très content",
    "Guten Morgen mein Freund",
    "Hello how are you today",
    "Привет как дела",
    "مرحبا كيف حالك"
]

for text in examples:
    print(text, "->", predict_language(text))

## Conclusion

In this project, a language detection model was developed using NLP techniques.

Text preprocessing steps such as cleaning, tokenization and stopword removal were applied. 
N-gram vectorization was used to transform text into numerical features.

A Logistic Regression classifier was trained to identify 17 different languages.
The model achieved approximately **95% accuracy**, showing strong performance in detecting languages from text samples.

This approach can be applied in multilingual systems, translation tools and text processing pipelines.

In [ ]:
import joblib, os

os.makedirs("src", exist_ok=True)
joblib.dump(model, "src/language_model.pkl")
joblib.dump(vect, "src/vectorizer.pkl")

print("Saved to src/: language_model.pkl, vectorizer.pkl")